# Lab: Inference Benchmarking on AMD MI300X

In this lab, you'll verify your vLLM deployment and establish baseline performance metrics for the GPT-OSS 120B reasoning model running on AMD Instinct MI300X.

**Model:** `openai/gpt-oss-120b`  
**Hardware:** AMD Instinct MI300X (192GB VRAM)  
**Framework:** vLLM with ROCm and AITER optimizations

## 1. Using AMD Developer Cloud?

Pre-built AMD Developer Cloud droplets include:
- GPT-OSS 120B pre-loaded 
- Optimized vLLM configuration for MI300X
- All AITER environment variables pre-set

**If you're on a pre-built droplet**, skip directly to Section 3.

**In this notebook**, we'll walk through the full deployment process—launching vLLM manually so you understand how to configure and optimize model serving on MI300X hardware.

## 2. Launch vLLM Server

With the container ready, the next step is launching the vLLM server. **Open a new terminal** and run the following script:
```bash
export VLLM_ROCM_USE_AITER=1
export VLLM_ROCM_QUICK_REDUCE_QUANTIZATION=INT4
export VLLM_ROCM_USE_AITER_MHA=0
export HIP_FORCE_DEV_KERNARG=1

vllm serve openai/gpt-oss-120b \
    --tensor-parallel-size 1 \
    --max-model-len 8192 \
    --trust-remote-code \
    --dtype bfloat16 \
    --kv-cache-dtype fp8 \
    --gpu-memory-utilization 0.92 \
    --no-enable-chunked-prefill \
    --max-num-seqs 512 \
    --port 8080
```

On MI300 GPUs, we enable AITER kernels and unified attention. We then start the GPT-OSS 120B model using the `vllm serve` command. 

Once launched, your GPU runs a production-ready inference server—essentially turning it into a private mini-ChatGPT.

After a few minutes, if you see logs like below, the vLLM server is ready:
```
INFO:     Started server process [1022120]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
```

### Review ROCm and AITER Environment Variables

The performance of vLLM on AMD GPUs depends heavily on environment variables that configure ROCm and AITER (AMD Inference Tensor Engine for ROCm).

**Key variables to look for:**

| Variable | Purpose |
|----------|---------|
| `VLLM_ROCM_USE_AITER=1` | Master switch — forces vLLM to use AITER's optimized C++ kernels instead of standard Python attention layers |
| `VLLM_ROCM_USE_AITER_MHA=0` | Disables AITER's multi-head attention (still evolving, kept off for stability) |
| `VLLM_ROCM_QUICK_REDUCE_QUANTIZATION=INT4` | Uses 4-bit integers to maximize memory bandwidth |
| `HIP_FORCE_DEV_KERNARG=1` | Micro-optimization for faster kernel argument passing |


## 3. Test the vLLM API

The vLLM server exposes an OpenAI-compatible API on port 8080. Let's verify the server is responding and test a simple chat completion.

In [ ]:
import requests

BASE_URL = "http://localhost:8080/v1"

### Query Available Models

Retrieve the list of models served by vLLM. In this droplet, there's only one model loaded: `openai/gpt-oss-120b`.

In [ ]:
response = requests.get(f"{BASE_URL}/models")
models = response.json()

MODEL_NAME = models["data"][0]["id"]
print(f"Using model: {MODEL_NAME}")

### Test Chat Completion

Run a simple inference test to verify the model is responding correctly. 

In [ ]:
response = requests.post(
    f"{BASE_URL}/chat/completions",
    json={
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "How many r's are in the phrase Strawberry Milkshaker?"}
        ]
    }
)

result = response.json()
print(result["choices"][0]["message"]["content"])

## 4. Run the Benchmark

The benchmark suite is pre-installed. It sends **800 requests** with **50 concurrent connections** to measure inference throughput. This typically takes 1-2 minutes to complete.

**Benchmarking Scripts**: The scripts used in this lab are open source and available at [github.com/AMD-AGI/GPU-Optimization-for-LLM-Inference](https://github.com/AMD-AGI/GPU-Optimization-for-LLM-Inference).

In [ ]:
!cd GPU-Optimization-for-LLM-Inference && python summary_benchmark.py --port 8080 --model "openai/gpt-oss-120b"